# Online Appendix: EmbeddingGemma 2 Robustness Check

Writes `Appendix_Gemma.tex`, the section of the Online Appendix that compares the main analysis (fine-tuned DoubleMLDeep embeddings, `main/`) with the re-run using off-the-shelf EmbeddingGemma 2 embeddings (text + image, 256 dimensions, `gemma/`). All tables and numbers are read from the outputs of the executed notebooks:

| content | fine-tuned embeddings | EmbeddingGemma 2 |
| --- | --- | --- |
| model and inference settings | | `gemma/data/embeddings/embeddinggemma2_txtimg.json` |
| clusters, closest products | | `../outputs/01_1_create_dataset_txt_img/cluster_overview.csv`, `../outputs/02_cluster_centroid_products/collage_similarity_cluster_*.png` |
| predictive performance (test $R^2$) | `main/code/03_1_predictive_performance_txt_img.ipynb` | `03_1_predictive_performance_txt_img.ipynb` |
| homogeneous elasticities | `main/output/04_evaluation/homogeneous_elasticity_estimates.csv` | `../outputs/04_evaluation/homogeneous_elasticity_estimates.csv` |
| sensitivity analysis | `main/output/04_evaluation/sensitivity_*.csv` | `../outputs/04_evaluation/sensitivity_*.csv`, `key_results.json` |
| heterogeneous elasticities | `../outputs/04_evaluation_comparison/finetuned/{cate_estimates,chi2_tests,sorted_effects}.csv`, `main/output/04_evaluation/cluster_sorted_effects.png` | `../outputs/04_evaluation/{cate_estimates,chi2_tests,sorted_effects}.csv`, `cluster_sorted_effects.png` |

The observed-covariate benchmark of the fine-tuned embeddings is taken from the re-run `../outputs/04_evaluation_comparison/finetuned`, which is checked against `main/output/04_evaluation` below.

Requires `04_evaluation.ipynb` (default parameters and the `finetuned` run of `04_evaluation_comparison.ipynb`). Run from `gemma/code`.

In [ ]:
import os
import re
import json
import shutil
from html.parser import HTMLParser
from pathlib import Path

import numpy as np
import pandas as pd
import nbformat

# Parameters
output_dir = "../outputs/05_appendix_gemma"
paper_figure_dir = "paper/figures_revision/gemma"  # figure path
main_dir = "../../main/output/04_evaluation"
gemma_dir = "../outputs/04_evaluation"
finetuned_rerun_dir = "../outputs/04_evaluation_comparison/finetuned"
Path(output_dir, "figures").mkdir(parents=True, exist_ok=True)

## Inputs and consistency checks

In [2]:
meta = json.load(open("../data/embeddings/embeddinggemma2_txtimg.json"))

# homogeneous elasticities (Models I-1 to I-3)
ate_main = pd.read_csv(os.path.join(main_dir, "homogeneous_elasticity_estimates.csv"), index_col=0)
ate_gemma = pd.read_csv(os.path.join(gemma_dir, "homogeneous_elasticity_estimates.csv"), index_col=0)
assert list(ate_main.index) == list(ate_gemma.index)

# sensitivity analysis
sens_main = pd.read_csv(os.path.join(main_dir, "sensitivity_calibrated.csv"))
sens_gemma_main_shares = pd.read_csv(os.path.join(gemma_dir, "sensitivity_main_shares.csv"))
sens_gemma_own = pd.read_csv(os.path.join(gemma_dir, "sensitivity_calibrated.csv"))
rv_main = pd.read_csv(os.path.join(main_dir, "sensitivity_robustness_values.csv")).set_index("null_hypothesis")
rv_gemma = pd.read_csv(os.path.join(gemma_dir, "sensitivity_robustness_values.csv")).set_index("null_hypothesis")
assert np.allclose(sens_gemma_main_shares["share"], sens_main["share"], rtol=0, atol=0)
key_gemma = json.load(open(os.path.join(gemma_dir, "key_results.json")))["sensitivity"]
key_main = json.load(open(os.path.join(finetuned_rerun_dir, "key_results.json")))["sensitivity"]

# the re-run of the fine-tuned specification reproduces main/output/04_evaluation
for file_name in ["homogeneous_elasticity_estimates.csv", "sensitivity_calibrated.csv", "sensitivity_robustness_values.csv"]:
    a = pd.read_csv(os.path.join(main_dir, file_name)).select_dtypes("number")
    b = pd.read_csv(os.path.join(finetuned_rerun_dir, file_name)).select_dtypes("number")
    assert np.allclose(a.values, b.values, rtol=0, atol=1e-9), file_name
assert np.isclose(key_main["scaled_implied_rv"], sens_main.loc[0, "share"], rtol=0, atol=1e-12)
assert np.isclose(key_gemma["scaled_implied_rv"], sens_gemma_own.loc[0, "share"], rtol=0, atol=1e-12)
print("fine-tuned re-run reproduces main/output/04_evaluation")

fine-tuned re-run reproduces main/output/04_evaluation


In [3]:
class _TableParser(HTMLParser):
    """Minimal parser for the HTML representation of a pandas DataFrame."""

    def __init__(self):
        super().__init__()
        self.rows, self._row, self._cell = [], None, None

    def handle_starttag(self, tag, attrs):
        if tag == "tr":
            self._row = []
        elif tag in ("td", "th") and self._row is not None:
            self._cell = ""

    def handle_endtag(self, tag):
        if tag in ("td", "th") and self._cell is not None:
            self._row.append(self._cell.strip())
            self._cell = None
        elif tag == "tr" and self._row is not None:
            self.rows.append(self._row)
            self._row = None

    def handle_data(self, data):
        if self._cell is not None:
            self._cell += data


def read_result_table(notebook, variable):
    """Read the DataFrame displayed by the cell `<variable> = pd.concat(...)` of an executed notebook."""
    nb = nbformat.read(notebook, as_version=4)
    cells = [c for c in nb.cells if c.cell_type == "code" and re.match(rf"{variable} = pd\.concat", c.source.strip())]
    assert len(cells) == 1, (notebook, variable)
    html = [o["data"]["text/html"] for o in cells[0].outputs if "data" in o and "text/html" in o["data"]]
    assert len(html) == 1
    parser = _TableParser()
    parser.feed(html[0])
    header, *body = parser.rows
    df = pd.DataFrame([r[1:] for r in body], index=[r[0] for r in body], columns=header[1:]).astype(float)
    return df


r2 = {}
for label, notebook in [
    ("finetuned", "../../main/code/03_1_predictive_performance_txt_img.ipynb"),
    ("gemma", "03_1_predictive_performance_txt_img.ipynb"),
]:
    levels = read_result_table(notebook, "results_df")
    diffs = read_result_table(notebook, "results_df_diff")
    r2[label] = pd.DataFrame(
        {
            "Q": levels["R2 Q Test"],
            "P": levels["R2 P Test"],
            "dQ": diffs["R2 Q Test"],
            "dP": diffs["R2 P Test"],
        }
    )
# the tabular-only benchmarks do not depend on the embeddings
for row in ["OLS (Tabular)", "Boosting (Tabular)"]:
    assert np.allclose(r2["finetuned"].loc[row], r2["gemma"].loc[row], rtol=0, atol=1e-12)
pd.concat(r2, axis=1).round(4)

finetuned                           gemma  \
                                          Q       P      dQ      dP       Q   
OLS (Tabular)                        0.2084  0.1582  0.0708  0.0064  0.2084   
Boosting (Tabular)                   0.4751  0.1854  0.1513 -0.0043  0.4751   
OLS (Tabular + PCA)                  0.4647  0.6063  0.0706  0.0064  0.3105   
Boosting (Tabular + PCA)             0.5322  0.6436  0.1379 -0.0045  0.5464   
OLS (Tabular + Similarities)         0.4727  0.5702  0.0707  0.0064  0.3030   
Boosting (Tabular + Similarities)    0.5258  0.6253  0.1463 -0.0024  0.5320   
OLS (Tabular + Embeddings)           0.5124  0.6214  0.0547 -0.0027  0.4088   
Boosting (Tabular + Embeddings)      0.5528  0.6564  0.1011 -0.0119  0.5877   
Deep Time Independent                0.5044  0.6726  0.1089  0.0115  0.5044   
Deep Time Dependent                  0.5953  0.6669  0.1217  0.0165  0.5953   

                                                           
                                        P      dQ      dP  
OLS (Tabular)                      0.1582  0.0708  0.0064  
Boosting (Tabular)                 0.1854  0.1513 -0.0043  
OLS (Tabular + PCA)                0.1650  0.0709  0.0067  
Boosting (Tabular + PCA)           0.3756  0.1707  0.0047  
OLS (Tabular + Similarities)       0.1680  0.0709  0.0066  
Boosting (Tabular + Similarities)  0.3222  0.1680  0.0054  
OLS (Tabular + Embeddings)         0.5230  0.0707  0.0055  
Boosting (Tabular + Embeddings)    0.5554  0.1836  0.0156  
Deep Time Independent              0.6726  0.1089  0.0115  
Deep Time Dependent                0.6669  0.1217  0.0165

## Tables

In [4]:
def num(x, digits=3):
    """Number in math mode (proper minus sign)."""
    return f"${x:.{digits}f}$"


def pct(x, digits=2):
    """Test R^2 in percent; negative values are set to 0 as in the main text."""
    return f"{100 * max(x, 0.0):.{digits}f}"


# --- predictive performance -----------------------------------------------------------
r2_rows = [
    ("Linear Reg [all tabular]", "OLS (Tabular)"),
    ("Boosted Trees Reg [all tabular]", "Boosting (Tabular)"),
    None,
    ("Linear Reg [+5 PCAs]", "OLS (Tabular + PCA)"),
    ("Linear Reg [+5 Similarities]", "OLS (Tabular + Similarities)"),
    ("Linear Reg [+256 Embeddings]", "OLS (Tabular + Embeddings)"),
    ("Boosted Trees Reg [+5 PCAs]", "Boosting (Tabular + PCA)"),
    ("Boosted Trees Reg [+5 Similarities]", "Boosting (Tabular + Similarities)"),
    ("Boosted Trees Reg [+256 Embeddings]", "Boosting (Tabular + Embeddings)"),
]
lines = []
for row in r2_rows:
    if row is None:
        lines.append("\\midrule")
        continue
    label, key = row
    cells = []
    for target in ["Q", "P", "dQ", "dP"]:
        a, b = r2["finetuned"].loc[key, target], r2["gemma"].loc[key, target]
        if key in ("OLS (Tabular)", "Boosting (Tabular)"):
            cells.append(f"\\multicolumn{{2}}{{c}}{{{pct(a)}}}")
        else:
            cells.append(f"{pct(a)} & {pct(b)}")
    lines.append(f"{label} & " + " & ".join(cells) + "\\\\")
table_r2 = "\n".join(lines)

# --- homogeneous elasticities -----------------------------------------------------------
lines = []
for spec in ate_main.index:
    cells = []
    for df in [ate_main, ate_gemma]:
        r = df.loc[spec]
        cells.append(f"{num(r['coef'])} & {num(r['std err'])} & [{num(r['lower'])}, {num(r['upper'])}]")
    lines.append(f"{{{spec}}} & " + " & ".join(cells) + "\\\\")
table_ate = "\n".join(lines)


# --- sensitivity at the residual shares of the main analysis ----------------------------
def sensitivity_rows(df):
    return "\n".join(
        f"{r['scenario']} & {100 * r['share']:.2f}\\% & {num(r['theta_lower'])} & {num(r['theta_upper'])} "
        f"& {num(r['ci_lower'])} & {num(r['ci_upper'])}\\\\"
        for _, r in df.iterrows()
    )


table_sens_main = sensitivity_rows(sens_main)
table_sens_gemma = sensitivity_rows(sens_gemma_main_shares)

print(table_r2, table_ate, table_sens_main, table_sens_gemma, sep="\n\n")

Linear Reg [all tabular] & \multicolumn{2}{c}{20.84} & \multicolumn{2}{c}{15.82} & \multicolumn{2}{c}{7.08} & \multicolumn{2}{c}{0.64}\\
Boosted Trees Reg [all tabular] & \multicolumn{2}{c}{47.51} & \multicolumn{2}{c}{18.54} & \multicolumn{2}{c}{15.13} & \multicolumn{2}{c}{0.00}\\
\midrule
Linear Reg [+5 PCAs] & 46.47 & 31.05 & 60.63 & 16.50 & 7.06 & 7.09 & 0.64 & 0.67\\
Linear Reg [+5 Similarities] & 47.27 & 30.30 & 57.02 & 16.80 & 7.07 & 7.09 & 0.64 & 0.66\\
Linear Reg [+256 Embeddings] & 51.24 & 40.88 & 62.14 & 52.30 & 5.47 & 7.07 & 0.00 & 0.55\\
Boosted Trees Reg [+5 PCAs] & 53.22 & 54.64 & 64.36 & 37.56 & 13.79 & 17.07 & 0.00 & 0.47\\
Boosted Trees Reg [+5 Similarities] & 52.58 & 53.20 & 62.53 & 32.23 & 14.63 & 16.80 & 0.00 & 0.54\\
Boosted Trees Reg [+256 Embeddings] & 55.28 & 58.77 & 65.64 & 55.54 & 10.11 & 18.36 & 0.00 & 1.56\\

{I-1. Linear ($P_{t-1}$, $Q_{t-1}$)} & $-0.690$ & $0.040$ & [$-0.756$, $-0.624$] & $-0.690$ & $0.040$ & [$-0.756$, $-0.624$]\\
{I-1. Linear ($P_{t-1}$,